# Step1: Setup and Configuration

In [1]:
#Uncomment the following lines to install required packages if you are running this notebook in a new environment.
# Install required packages (this may take 2-3 minutes)
# Installing in compatible groups to avoid dependency conflicts
# %pip install --upgrade --quiet "google-cloud-aiplatform>=1.38.0" "kfp>=2.4.0"
# %pip install --upgrade --quiet "scikit-learn>=1.4.2" "imbalanced-learn>=0.12.0"
# %pip install --upgrade --quiet pandas numpy

# print("\n" + "="*70)
# print(" Installation complete!")
# print("="*70)
# print("\n NEXT STEPS:")
# print("1. Restart the kernel: Kernel → Restart Kernel")
# print("2. Skip this cell (don't run it again)")
# print("3. Continue from cell 2 (Import libraries)")
# print("="*70)

In [2]:
import google.auth
import google.cloud.aiplatform
import kfp
import sklearn
import imblearn
import pandas
import numpy

print("google-auth:", google.auth.__version__)
print("aiplatform:", google.cloud.aiplatform.__version__)
print("kfp:", kfp.__version__)
print("sklearn:", sklearn.__version__)
print("imblearn:", imblearn.__version__)
print("pandas:", pandas.__version__)
print("numpy:", numpy.__version__)

print(" Imports successful")

google-auth: 2.59.1
aiplatform: 2.3.0
kfp: 2.17.0
sklearn: 1.9.1
imblearn: 0.14.2
pandas: 3.0.6
numpy: 2.5.3
 Imports successful


## Importing Libraries and Initializing vertex AI

In [3]:
# Import libraries
import os
import json
from datetime import datetime
from typing import NamedTuple

from google.cloud import aiplatform
from kfp import dsl
from kfp.dsl import Output, Input, Dataset, Model, Metrics, component
from kfp import compiler
import pandas as pd
import numpy as np

# Configuration - UPDATE THESE VALUES
PROJECT_ID = "mlopstest-510202"
REGION = "northamerica-northeast1"           # Choose your preferred region
BUCKET_NAME = f"{PROJECT_ID}-vertex-pipelines"  # GCS bucket for artifacts
PIPELINE_ROOT = f"gs://{BUCKET_NAME}/pipeline_root"

# Initialize Vertex AI
aiplatform.init(project=PROJECT_ID, location=REGION)

print(f" Project ID: {PROJECT_ID}")
print(f" Region: {REGION}")
print(f" Pipeline Root: {PIPELINE_ROOT}")

 Project ID: mlopstest-510202
 Region: northamerica-northeast1
 Pipeline Root: gs://mlopstest-510202-vertex-pipelines/pipeline_root


## Create GCS bucket if it doesn't exist

In [4]:
!gsutil ls -b gs://{BUCKET_NAME} || gsutil mb -l {REGION} gs://{BUCKET_NAME}
print(f"Bucket ready: gs://{BUCKET_NAME}")

gs://mlopstest-510202-vertex-pipelines/
Bucket ready: gs://mlopstest-510202-vertex-pipelines


Grant service account permissions to access the bucket
Get the project number for the service account

In [5]:
!gcloud services enable cloudresourcemanager.googleapis.com --project={PROJECT_ID}

result = !gcloud projects describe {PROJECT_ID} --format="value(projectNumber)"
PROJECT_NUMBER = result[0] if result else None

if PROJECT_NUMBER:
    SERVICE_ACCOUNT = f"{PROJECT_NUMBER}-compute@developer.gserviceaccount.com"
    print(f"Granting permissions to service account: {SERVICE_ACCOUNT}")

    !gsutil iam ch serviceAccount:{SERVICE_ACCOUNT}:roles/storage.objectAdmin gs://{BUCKET_NAME}

    print("Permissions granted successfully!")
    print(f"Service account can now read/write to gs://{BUCKET_NAME}")
else:
    print("Could not determine project number.")

Granting permissions to service account: 288053996263-compute@developer.gserviceaccount.com
Permissions granted successfully!
Service account can now read/write to gs://mlopstest-510202-vertex-pipelines


No changes made to gs://mlopstest-510202-vertex-pipelines/


# Step 2: Define Pipeline Components
We'll create modular, reusable components for each step of our ML pipeline.

### Component 1: Data Generation

Generates 5,000 synthetic credit card transactions with realistic fraud patterns.

In [6]:
@component(
    base_image="python:3.10",
    packages_to_install=["pandas", "numpy", "scikit-learn"]
)
def generate_credit_card_data(
    dataset: Output[Dataset],
    num_samples: int = 5000,
    fraud_rate: float = 0.025
):
    """Generate synthetic credit card transaction data with fraud patterns."""
    import pandas as pd
    import numpy as np
    from datetime import datetime, timedelta
    
    np.random.seed(42)
    
    # Generate transaction data
    num_fraud = int(num_samples * fraud_rate)
    num_legitimate = num_samples - num_fraud
    
    # Legitimate transactions
    legitimate_amounts = np.random.lognormal(mean=3.5, sigma=1.2, size=num_legitimate)
    legitimate_amounts = np.clip(legitimate_amounts, 1, 500)
    
    # Fraudulent transactions (typically unusual patterns)
    fraud_amounts = np.concatenate([
        np.random.uniform(500, 2000, size=num_fraud//2),  # High amounts
        np.random.uniform(1, 10, size=num_fraud - num_fraud//2)  # Very small amounts
    ])
    
    # Create DataFrame
    data = {
        'amount': np.concatenate([legitimate_amounts, fraud_amounts]),
        'transaction_hour': np.concatenate([
            np.random.choice(range(6, 23), size=num_legitimate),  # Normal hours
            np.random.choice(range(0, 6), size=num_fraud)  # Late night fraud
        ]),
        'merchant_category': np.concatenate([
            np.random.choice(['grocery', 'restaurant', 'gas', 'retail'], size=num_legitimate),
            np.random.choice(['electronics', 'jewelry', 'online'], size=num_fraud)
        ]),
        'distance_from_home': np.concatenate([
            np.random.exponential(scale=10, size=num_legitimate),
            np.random.uniform(50, 500, size=num_fraud)
        ]),
        'days_since_last_transaction': np.concatenate([
            np.random.poisson(lam=3, size=num_legitimate),
            np.random.uniform(0, 1, size=num_fraud)
        ]),
        'online_transaction': np.concatenate([
            np.random.choice([0, 1], size=num_legitimate, p=[0.7, 0.3]),
            np.random.choice([0, 1], size=num_fraud, p=[0.2, 0.8])
        ]),
        'is_fraud': np.concatenate([
            np.zeros(num_legitimate),
            np.ones(num_fraud)
        ])
    }
    
    df = pd.DataFrame(data)
    
    # Shuffle the data
    df = df.sample(frac=1, random_state=42).reset_index(drop=True)
    
    # Add transaction IDs
    df.insert(0, 'transaction_id', range(1, len(df) + 1))
    
    # Save to output
    df.to_csv(dataset.path, index=False)
    
    print(f"     Generated {len(df)} transactions")
    print(f"   - Legitimate: {num_legitimate} ({num_legitimate/len(df)*100:.1f}%)")
    print(f"   - Fraudulent: {num_fraud} ({num_fraud/len(df)*100:.1f}%)")
    print(f"   - Saved to: {dataset.path}")


### Component 2: Data Validation
Validates data schema, checks for missing values, and verifies data quality.

In [7]:
@component(
    base_image="python:3.10",
    packages_to_install=["pandas", "numpy"]
)
def validate_data(
    dataset: Input[Dataset],
    validated_dataset: Output[Dataset],
    metrics: Output[Metrics]
) -> NamedTuple('Outputs', [('validation_passed', bool), ('num_records', int)]):
    """Validate data quality and schema."""
    import pandas as pd
    from collections import namedtuple
    
    # Load data
    df = pd.read_csv(dataset.path)
    
    # Validation checks
    required_columns = ['transaction_id', 'amount', 'transaction_hour', 
                       'merchant_category', 'distance_from_home', 
                       'days_since_last_transaction', 'online_transaction', 'is_fraud']
    
    validation_passed = True
    issues = []
    
    # Check 1: Required columns present
    missing_cols = set(required_columns) - set(df.columns)
    if missing_cols:
        issues.append(f"Missing columns: {missing_cols}")
        validation_passed = False
    
    # Check 2: No missing values
    if df.isnull().sum().sum() > 0:
        issues.append(f"Missing values found: {df.isnull().sum().to_dict()}")
        validation_passed = False
    
    # Check 3: Valid data ranges
    if (df['amount'] < 0).any():
        issues.append("Negative amounts found")
        validation_passed = False
    
    if (df['transaction_hour'] < 0).any() or (df['transaction_hour'] > 23).any():
        issues.append("Invalid transaction hours")
        validation_passed = False
    
    # Check 4: Fraud ratio
    fraud_rate = df['is_fraud'].mean()
    if fraud_rate < 0.01 or fraud_rate > 0.1:
        issues.append(f"Unusual fraud rate: {fraud_rate:.2%}")
    
    # Log metrics
    metrics.log_metric('total_records', len(df))
    metrics.log_metric('fraud_rate', fraud_rate)
    metrics.log_metric('validation_passed', 1.0 if validation_passed else 0.0)
    
    if validation_passed:
        print(" Data validation PASSED")
        print(f"   - Total records: {len(df)}")
        print(f"   - Fraud rate: {fraud_rate:.2%}")
        df.to_csv(validated_dataset.path, index=False)
    else:
        print(" Data validation FAILED")
        for issue in issues:
            print(f"   - {issue}")
        raise ValueError(f"Data validation failed: {', '.join(issues)}")
    
    outputs = namedtuple('Outputs', ['validation_passed', 'num_records'])
    return outputs(validation_passed, len(df))

### Component 3: Data Preprocessing
Performs feature engineering, encoding, and handles class imbalance with SMOTE.

In [8]:
@component(
    base_image="python:3.10",
    packages_to_install=["pandas", "numpy", "scikit-learn", "imbalanced-learn"]
)
def preprocess_data(
    dataset: Input[Dataset],
    train_dataset: Output[Dataset],
    test_dataset: Output[Dataset],
    test_size: float = 0.2,
    apply_smote: bool = True
):
    """Preprocess data with encoding, scaling, and SMOTE for class balance."""
    import pandas as pd
    import numpy as np
    from sklearn.model_selection import train_test_split
    from sklearn.preprocessing import StandardScaler, LabelEncoder
    from imblearn.over_sampling import SMOTE
    
    # Load data
    df = pd.read_csv(dataset.path)
    
    # Feature engineering
    df['amount_per_day'] = df['amount'] / (df['days_since_last_transaction'] + 1)
    df['is_weekend_hour'] = ((df['transaction_hour'] < 6) | (df['transaction_hour'] > 22)).astype(int)
    
    # Encode categorical features
    le = LabelEncoder()
    df['merchant_category_encoded'] = le.fit_transform(df['merchant_category'])
    
    # Select features
    feature_cols = ['amount', 'transaction_hour', 'distance_from_home', 
                   'days_since_last_transaction', 'online_transaction',
                   'merchant_category_encoded', 'amount_per_day', 'is_weekend_hour']
    
    X = df[feature_cols]
    y = df['is_fraud']
    
    # Split data
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=test_size, random_state=42, stratify=y
    )
    
    # Scale features
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)
    
    # Apply SMOTE to balance classes (training only)
    if apply_smote:
        smote = SMOTE(random_state=42)
        X_train_scaled, y_train = smote.fit_resample(X_train_scaled, y_train)
        print(f"   Applied SMOTE")
        print(f"   - Original class distribution: {dict(pd.Series(y).value_counts())}")
        print(f"   - Resampled class distribution: {dict(pd.Series(y_train).value_counts())}")
    
    # Create DataFrames with scaled features
    train_df = pd.DataFrame(X_train_scaled, columns=feature_cols)
    train_df['is_fraud'] = y_train.reset_index(drop=True)
    
    test_df = pd.DataFrame(X_test_scaled, columns=feature_cols)
    test_df['is_fraud'] = y_test.reset_index(drop=True)
    
    # Save datasets
    train_df.to_csv(train_dataset.path, index=False)
    test_df.to_csv(test_dataset.path, index=False)
    
    print(f"   Preprocessing complete")
    print(f"   - Training samples: {len(train_df)}")
    print(f"   - Testing samples: {len(test_df)}")
    print(f"   - Features: {len(feature_cols)}")


    

### Component 4: Model Training
Trains a Random Forest classifier for fraud detection.

In [9]:
@component(
    base_image="python:3.10",
    packages_to_install=["pandas", "numpy", "scikit-learn", "joblib"]
)
def train_model(
    train_dataset: Input[Dataset],
    model: Output[Model],
    n_estimators: int = 100,
    max_depth: int = 10
):
    """Train Random Forest classifier for fraud detection."""
    import pandas as pd
    import numpy as np
    from sklearn.ensemble import RandomForestClassifier
    import joblib
    
    # Load training data
    train_df = pd.read_csv(train_dataset.path)
    
    X_train = train_df.drop('is_fraud', axis=1)
    y_train = train_df['is_fraud']
    
    # Train model
    print(f" Training Random Forest...")
    print(f"   - n_estimators: {n_estimators}")
    print(f"   - max_depth: {max_depth}")
    print(f"   - Training samples: {len(X_train)}")
    
    rf_model = RandomForestClassifier(
        n_estimators=n_estimators,
        max_depth=max_depth,
        random_state=42,
        n_jobs=-1,
        class_weight='balanced'
    )
    
    rf_model.fit(X_train, y_train)
    
    # Save model
    model_path = model.path
    joblib.dump(rf_model, model_path)
    
    # Feature importance
    feature_importance = pd.DataFrame({
        'feature': X_train.columns,
        'importance': rf_model.feature_importances_
    }).sort_values('importance', ascending=False)
    
    print(f" Model training complete")
    print(f"\n Feature Importance:")
    for idx, row in feature_importance.head(5).iterrows():
        print(f"   {row['feature']}: {row['importance']:.4f}")

### Component 5: Model Evaluation
Evaluates model performance on test data with comprehensive metrics.

In [10]:
@component(
    base_image="python:3.10",
    packages_to_install=["pandas", "numpy", "scikit-learn", "joblib"]
)
def evaluate_model(
    model: Input[Model],
    test_dataset: Input[Dataset],
    metrics: Output[Metrics]
) -> NamedTuple('Outputs', [('accuracy', float), ('precision', float), ('recall', float), ('f1_score', float)]):
    """Evaluate model performance on test data."""
    import pandas as pd
    import numpy as np
    import joblib
    from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
    from sklearn.metrics import classification_report, confusion_matrix
    from collections import namedtuple
    
    # Load model and test data
    model_path = model.path
    rf_model = joblib.load(model_path)
    test_df = pd.read_csv(test_dataset.path)
    
    X_test = test_df.drop('is_fraud', axis=1)
    y_test = test_df['is_fraud']
    
    # Make predictions
    y_pred = rf_model.predict(X_test)
    y_pred_proba = rf_model.predict_proba(X_test)[:, 1]
    
    # Calculate metrics
    accuracy = accuracy_score(y_test, y_pred)   # Example: TP + TN / Total
    precision = precision_score(y_test, y_pred)
    recall = recall_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred)
    
    # Log metrics
    metrics.log_metric('accuracy', accuracy)
    metrics.log_metric('precision', precision)
    metrics.log_metric('recall', recall)
    metrics.log_metric('f1_score', f1)
    
    # Confusion matrix
    cm = confusion_matrix(y_test, y_pred)
    tn, fp, fn, tp = cm.ravel()
    
    metrics.log_metric('true_negatives', float(tn))
    metrics.log_metric('false_positives', float(fp))
    metrics.log_metric('false_negatives', float(fn))
    metrics.log_metric('true_positives', float(tp))
    
    # Print results
    print(f" Model Evaluation Complete")
    print(f"\n Performance Metrics:")
    print(f"   - Accuracy:  {accuracy:.4f}")
    print(f"   - Precision: {precision:.4f}")
    print(f"   - Recall:    {recall:.4f}")
    print(f"   - F1-Score:  {f1:.4f}")
    print(f"\n Confusion Matrix:")
    print(f"   - True Negatives:  {tn}")
    print(f"   - False Positives: {fp}")
    print(f"   - False Negatives: {fn}")
    print(f"   - True Positives:  {tp}")
    
    outputs = namedtuple('Outputs', ['accuracy', 'precision', 'recall', 'f1_score'])
    return outputs(accuracy, precision, recall, f1)

### Component 6: Deployment Decision
Decides whether to deploy model based on performance thresholds.

In [11]:
@component(
    base_image="python:3.10"
)
def deploy_model(
    model: Input[Model],
    accuracy: float,
    min_accuracy: float = 0.90
) -> str:
    """Deploy model if it meets performance criteria."""
    
    if accuracy >= min_accuracy:
        print(f"   DEPLOYMENT APPROVED")
        print(f"   - Accuracy: {accuracy:.4f}")
        print(f"   - Threshold: {min_accuracy:.4f}")
        print(f"   - Model ready for production deployment")
        return "DEPLOYED"
    else:
        print(f"    DEPLOYMENT REJECTED")
        print(f"   - Accuracy: {accuracy:.4f}")
        print(f"   - Threshold: {min_accuracy:.4f}")
        print(f"   - Model needs improvement")
        return "NOT_DEPLOYED"

## Step 3: Define the Pipeline
Now we'll orchestrate all components into a single end-to-end pipeline.

In [ ]:
@dsl.pipeline(
    name="credit-card-fraud-detection-RandomForest",
    description="End-to-end MLOps pipeline for credit card fraud detection",
    pipeline_root=PIPELINE_ROOT,
)
def fraud_detection_pipeline(
    num_samples: int = 5000,
    fraud_rate: float = 0.025,
    test_size: float = 0.2,
    n_estimators: int = 100,
    max_depth: int = 10,
    min_accuracy: float = 0.90
):
    """
    Complete fraud detection pipeline with the following steps:
    1. Generate synthetic transaction data
    2. Validate data quality
    3. Preprocess and balance data with SMOTE
    4. Train Random Forest model
    5. Evaluate model performance
    6. Conditional deployment based on accuracy
    """
    
    # # Step 1: Generate data
    # generate_task = generate_credit_card_data(
    #     num_samples=num_samples,
    #     fraud_rate=fraud_rate
    # )
    
    generate_task = generate_credit_card_data(
    num_samples=num_samples,
    fraud_rate=fraud_rate
    ).set_cpu_limit("1")
    
    # Step 2: Validate data
    validate_task = validate_data(
        dataset=generate_task.outputs['dataset']
    )
    
    # Step 3: Preprocess data
    preprocess_task = preprocess_data(
        dataset=validate_task.outputs['validated_dataset'],
        test_size=test_size,
        apply_smote=True
    )
    
    # Step 4: Train model
    train_task = train_model(
        train_dataset=preprocess_task.outputs['train_dataset'],
        n_estimators=n_estimators,
        max_depth=max_depth
    )
    
    # Step 5: Evaluate model
    evaluate_task = evaluate_model(
        model=train_task.outputs['model'],
        test_dataset=preprocess_task.outputs['test_dataset']
    )
    
    # Step 6: Deploy model (conditional on accuracy)
    deploy_task = deploy_model(
        model=train_task.outputs['model'],
        accuracy=evaluate_task.outputs['accuracy'],
        min_accuracy=min_accuracy
    )

print(" Pipeline definition created")

 Pipeline definition created


## Step4: Compile the pipeline
Compile the pipeline to YAML (KFP v2 uses YAML format)

In [13]:
# Compile pipeline to YAML (KFP v2 uses YAML format)
import os
PIPELINE_FILE = "fraud_detection_pipeline.yaml"

# Remove old files if they exist
for old_file in [PIPELINE_FILE, "fraud_detection_pipeline.json"]:
    if os.path.exists(old_file):
        os.remove(old_file)
        print(f" Removed old file: {old_file}")

# Compile pipeline
compiler.Compiler().compile(
    pipeline_func=fraud_detection_pipeline,
    package_path=PIPELINE_FILE
)

# Verify the file was created
if os.path.exists(PIPELINE_FILE):
    file_size = os.path.getsize(PIPELINE_FILE)
    print(f"  Pipeline compiled successfully")
    print(f"   - Output file: {PIPELINE_FILE}")
    print(f"   - File size: {file_size:,} bytes")
else:
    print(f" Pipeline compilation failed!")

 Removed old file: fraud_detection_pipeline.yaml
  Pipeline compiled successfully
   - Output file: fraud_detection_pipeline.yaml
   - File size: 30,164 bytes


## Step 5: Execute the Pipeline
Submit the pipeline to Vertex AI for execution.

In [14]:
# Set pipeline parameters
pipeline_params = {
    'num_samples': 5000,
    'fraud_rate': 0.025,
    'test_size': 0.2,
    'n_estimators': 100,
    'max_depth': 10,
    'min_accuracy': 0.90
}

# Create and run pipeline job
job = aiplatform.PipelineJob(
    display_name="fraud-detection-pipeline-run",
    template_path=PIPELINE_FILE,
    pipeline_root=PIPELINE_ROOT,
    parameter_values=pipeline_params,
    enable_caching=True,#set to false into production
)

print(" Submitting pipeline to Vertex AI...")
job.submit()

print(f" Pipeline submitted successfully!")
print(f"   - Job name: {job.display_name}")
print(f"   - Job resource: {job.resource_name}")
print(f"\n Monitor progress in Cloud Console:")
print(f"   {job._dashboard_uri()}")

 Submitting pipeline to Vertex AI...
Creating PipelineJob
PipelineJob created. Resource name: projects/288053996263/locations/northamerica-northeast1/pipelineJobs/credit-card-fraud-detection-randomforest-20261004205421
To use this PipelineJob in another session:
pipeline_job = aiplatform.PipelineJob.get('projects/288053996263/locations/northamerica-northeast1/pipelineJobs/credit-card-fraud-detection-randomforest-20261004205421')
View Pipeline Job:
https://console.cloud.google.com/vertex-ai/locations/northamerica-northeast1/pipelines/runs/credit-card-fraud-detection-randomforest-20261004205421?project=288053996263
 Pipeline submitted successfully!
   - Job name: fraud-detection-pipeline-run
   - Job resource: projects/288053996263/locations/northamerica-northeast1/pipelineJobs/credit-card-fraud-detection-randomforest-20261004205421

 Monitor progress in Cloud Console:
   https://console.cloud.google.com/vertex-ai/locations/northamerica-northeast1/pipelines/runs/credit-card-fraud-detecti